# The Watchline agent and chat

*Executed 2026-11-22 against the live Watchline graph (PLUTO 26v1, HPD registrations through 2026-05-29), Geosupport 25b and
`claude-haiku-5-5`. The model's wording changes from run to run; the data under it does not. Re-run with `make notebook-check`
(needs Neo4j, the sidecar and `ANTHROPIC_API_KEY`; each run makes about a dozen cheap model calls).*

`make chat` starts a conversation with an agent that has **one tool**, `building_profile`. This notebook drives that same
agent, with the same functions the chat uses (`build_agent`, `ask`), and shows for every question:

1. what you typed,
2. the tool call the model chose (`-> building_profile(...)`),
3. the answer: a `kind`, a summary, facts that point at the data they rest on, and the caveats and sources, which **code** attaches from
   the tool result so the model cannot drop one. The terminal shows it as markdown; a later web page can render the same object.

Nothing here is a finding about a landlord. Violations, problems and litigation are what was **cited, reported or brought**.
The evals that grade answers like these automatically are `make eval-agent`; this notebook is for reading, not grading.

## Setup

`connect()` is the chat's own startup: it checks the Neo4j login and the sidecar and fails with one clear line if either is down.

In [1]:
import json
import uuid

from IPython.display import Markdown, display

from watchline.agent.agent import build_agent, memory_checkpointer, system_prompt
from watchline.agent.chat import ask, connect
from watchline.agent.config import AGENT_MODEL

client, graph, pg, close = connect()
agent = build_agent(client, graph, pg, checkpointer=memory_checkpointer())
RUN = uuid.uuid4().hex[:8]
print("model:", AGENT_MODEL)

model: claude-haiku-5-5


### What the model is given

The system prompt is written in `watchline/agent/prompts/system.md`. The ontology rules are filled in from
`watchline/agent/prompts/ontology_rules.md`, the same file `CLAUDE.md` imports, so Claude Code and the agent read the same words.

In [2]:
prompt = system_prompt()
print(f"{len(prompt.splitlines())} lines. The parts about reading the tool result and how to answer:\n")
start = prompt.index("## Reading the tool result")
print(prompt[start:prompt.index("## Rules for every answer")])

82 lines. The parts about reading the tool result and how to answer:

## Reading the tool result

Check `status` first, then `resolution.outcome`.

- `needs_borough`: ask the user for the borough or ZIP. Do not guess.
- `invalid_input`: say you need a house number and street, and ask for one.
- `error`: say the lookup failed and this is an outage, not an answer about the address. Suggest trying again.
- `ok` with `ambiguous`: list the candidate streets and ask which was meant. No lot was chosen.
- `ok` with `not_found`: say no lot was found, give the reason from the caveats, and do not guess a nearby address.
- `ok` with `exact` or `variant`: profile the lot, unless `lot` is null. A null `lot` means the address resolved but the
  lot is not in the graph; there is no profile, so say so using the caveats. For a variant, say what was typed and what the
  city matched. If the resolution is inferred, say so.

## How to answer

Your final message is structured: a `kind`, a `summary` and a li

### One helper

`chat(text, thread)` is the chat loop for one message. A *thread* is one conversation: messages on the same thread see each
other, which is how a follow-up like "what about 218?" works. It returns the turn, so the raw tool data can be inspected.

In [3]:
LOG = []


def chat(text, thread):
    config = {"configurable": {"thread_id": f"{RUN}-{thread}"}}
    print(f"you> {text}")
    turn = ask(agent, text, config, on_tool_call=lambda name, args: print(f"  -> {name}({json.dumps(args)})"))
    LOG.append((text, turn))
    display(Markdown(turn.answer))
    return turn


def tool_view(turn):
    """The parts of the first tool result that matter, to compare with the answer."""
    out = turn.tool_results[0]
    res = out["resolution"]
    print(f"status: {out['status']}   outcome: {res['outcome']}   bbl: {res['bbl']}   in_graph: {res['in_graph']}")
    print("counts:", {c["name"]: c["value"] for c in out.get("counts", [])})
    print("BINs:", [(b["bin"], "addressed" if b["addressed"] else "") for b in out.get("buildings", [])])
    for c in out["caveats"]:
        print("caveat:", c)

## 1. A lot with three buildings, then a follow-up on the same thread

`224 54 Street, Brooklyn` is one tax lot with three HPD-registered buildings. The model should call the tool once, name the
addressed building, list the others, and keep residential units as a **lot-wide** figure.

In [4]:
multi = chat("What can you tell me about 224 54 Street in Brooklyn?", "multi")

you> What can you tell me about 224 54 Street in Brooklyn?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


224 54 Street in Brooklyn resolved exactly to tax lot (BBL) 3008210012. The lot is a C1 building built in 1911 with 18 residential units, and HPD lists 3 buildings for it; the addressed building is BIN 3324451.

**What the records show**
- The address 224 54 Street resolved as an exact match to 224 54 STREET, tax lot (BBL) 3008210012, in Brooklyn. _(resolution)_
- The lot's PLUTO characteristics are: year built 1911, building class C1, and assessed total value 814500 (whole dollars, lot-wide). _(characteristics)_
- The lot has 18 residential units, lot-wide, per PLUTO (release 26v1); units belong to the lot, not to any building. _(characteristics)_
- PLUTO counts 3 buildings on the lot, and HPD registrations list 3 buildings (BINs) for it. _(counts)_
- BIN 3324451 (224 54TH STREET) is the addressed building. It has no violations listed; complaint problems are door/window 3, flooring/stairs 1, heat/hot water 3, paint/plaster 1, unsanitary condition 1, and water leak 1; no litigation; and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324451)_
- BIN 3324448 (218 54TH STREET) is not the addressed building. It has no violations, no complaint problems, no litigation, and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324448)_
- BIN 3324450 (220 54TH STREET) is not the addressed building. It has violations in class A (1, status Close), class B (1, status Close) and class C (1, status Close); complaint problems are door/window 2, heat/hot water 9, heat/hot water flagged as duplicate 2, and plumbing 1; no litigation; and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324450)_
- Across the lot's listed buildings, the violations are class A (1, Close), class B (1, Close) and class C (1, Close), and no litigation is listed. _(totals)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

The data the model was given, to check the answer against:

In [5]:
tool_view(multi)
data = multi.tool_results[0]
print()
print("BBL in the answer:", data["resolution"]["bbl"] in multi.answer)
for b in data["buildings"]:
    print(f"BIN {b['bin']} in the answer:", b["bin"] in multi.answer)

status: ok   outcome: exact   bbl: 3008210012   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 3, 'buildings (HPD)': 3, 'residential units': 18}
BINs: [('3324451', 'addressed'), ('3324448', ''), ('3324450', '')]
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'duplicate_flag_split', 'text': 'Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.'}

BBL in the answer: True
BIN 3324451 in the answer: True
BIN 3324448 in the answer: True
BIN 3324450 in the answer: True


### The answer as data

`turn.structured` is the object behind that markdown. The model wrote only the `kind`, the summary and the facts (with pointers
into the tool result). Everything else was filled in by code: the evidence (the tool's own data, copied in), the caveats and the
sources.

In [6]:
a = multi.structured
print("kind:", a.kind)
print("caveat codes:", sorted(a.caveat_codes))
print("sources:", a.sources)
print()
for f in a.facts[:4]:
    print("-", f.statement[:110])
    print("    evidence:", [(e.section, e.bin) for e in f.evidence], " unresolved:", f.unresolved)
print()
print("evidence data for the first fact (copied from the tool result, not typed by the model):")
print(json.dumps(a.facts[0].evidence[0].data, indent=1, default=str)[:600])

kind: profile
caveat codes: ['cited_not_fault', 'duplicate_flag_split', 'lot_wide']
sources: ['Geosupport (release 25b)', 'PLUTO (release 26v1)', 'HPD registrations (data through 2026-05-29)']

- The address 224 54 Street resolved as an exact match to 224 54 STREET, tax lot (BBL) 3008210012, in Brooklyn.
    evidence: [('resolution', None)]  unresolved: []
- The lot's PLUTO characteristics are: year built 1911, building class C1, and assessed total value 814500 (whol
    evidence: [('characteristics', None)]  unresolved: []
- The lot has 18 residential units, lot-wide, per PLUTO (release 26v1); units belong to the lot, not to any buil
    evidence: [('characteristics', None)]  unresolved: []
- PLUTO counts 3 buildings on the lot, and HPD registrations list 3 buildings (BINs) for it.
    evidence: [('counts', None)]  unresolved: []

evidence data for the first fact (copied from the tool result, not typed by the model):
{
 "input": "224 54 Street",
 "outcome": "exact",
 "bbl": "300821001

The follow-up is answered on the **same thread**. Because the first answer already holds all three buildings, the model may answer
from the conversation or call the tool again; either is fine, as long as 218 54 Street is placed on the same tax lot.

In [7]:
follow = chat("And what about the building at 218?", "multi")
print("tool calls this turn:", [c["args"] for c in follow.tool_calls] or "none (answered from the conversation)")

you> And what about the building at 218?


The 218 54th Street building is BIN 3324448, one of the three buildings HPD lists for tax lot 3008210012. It is not the addressed building, and the records show no violations, no complaint problems and no litigation for it.

**What the records show**
- BIN 3324448 is recorded as 218 54TH STREET and is not the addressed building; it is one of the 3 buildings HPD lists for tax lot 3008210012. _(buildings BIN 3324448; counts)_
- BIN 3324448 has no violations, no complaint problems and no litigation listed, and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324448)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

tool calls this turn: none (answered from the conversation)


## 2. Two building counts that disagree

PLUTO says 2 buildings at `17 Battery Place`; HPD lists 1. Neither is "the number of buildings". The answer should give both, each
labelled with its source.

In [8]:
battery = chat("How many buildings are at 17 Battery Place, Manhattan?", "battery")
tool_view(battery)

you> How many buildings are at 17 Battery Place, Manhattan?


  -> building_profile({"address": "17 Battery Place", "borough": "Manhattan"})


The records give two building counts for 17 Battery Place, Manhattan: PLUTO counts 2 buildings on the lot, and HPD registrations list 1 building. These are different sources and neither alone is the definitive number of buildings.

**What the records show**
- The address resolved exactly to tax lot (BBL) 1000157501, matched as 17 BATTERY PLACE. _(resolution)_
- PLUTO (release 26v1) counts 2 buildings on the lot. _(counts)_
- HPD registrations list 1 building (BIN) for the lot. _(counts)_
- The one HPD-listed building is BIN 1082634, recorded at 1 WEST STREET; it is the building the address resolved to. _(buildings BIN 1082634)_
- The lot has 500 residential units, lot-wide (PLUTO release 26v1); this figure belongs to the lot, not to the building. _(characteristics)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Two building counts: PLUTO says 2, HPD registrations list 1. They are different sources and differ on many lots; neither alone is 'the number of buildings'.
- Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings either way.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

status: ok   outcome: exact   bbl: 1000157501   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 2, 'buildings (HPD)': 1, 'residential units': 500}
BINs: [('1082634', 'addressed')]
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'building_counts_differ', 'text': "Two building counts: PLUTO says 2, HPD registrations list 1. They are different sources and differ on many lots; neither alone is 'the number of buildings'."}
caveat: {'code': 'harassment_finding_definition', 'text': "Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings e

## 3. A lot HPD does not register

`70 Broad Street` is a tax lot with no HPD-registered building. The records hold no violations, complaints or litigation for it,
**because they exist only for registered buildings**. That is not the same as "none". The answer should say "no building recorded in
the HPD registrations", not "no buildings" and not "no violations".

In [9]:
broad = chat("Tell me about 70 Broad Street, Manhattan.", "broad")
tool_view(broad)

you> Tell me about 70 Broad Street, Manhattan.


  -> building_profile({"address": "70 Broad Street", "borough": "Manhattan"})


70 Broad Street, Manhattan resolved exactly to tax lot (BBL) 1000110015. No building is recorded in the HPD registrations for this lot, so no violations, complaint problems or litigation can be read for it.

**What the records show**
- The address 70 Broad Street resolved as an exact match to 70 BROAD STREET, tax lot (BBL) 1000110015, and the lot is in the graph. _(resolution)_
- PLUTO (release 26v1) records the lot's year built as 1908 and its building class as O8. _(characteristics)_
- PLUTO (release 26v1) records 3 residential units for the lot as a whole; this is a lot-wide figure, not a per-building figure. _(characteristics)_
- PLUTO (release 26v1) records the lot's assessed total value as 4,751,550 whole dollars; this is a lot-wide figure. _(characteristics)_
- PLUTO (release 26v1) counts 1 building on the lot. _(counts)_
- The HPD registrations list 0 buildings (BINs) for this lot, so the lot is not HPD-registered. _(counts; buildings)_
- No building is recorded in the HPD registrations for this lot, so no violations, complaint problems or litigation can be read for it; these exist only for HPD-registered buildings. _(buildings; totals)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

status: ok   outcome: exact   bbl: 1000110015   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 1, 'buildings (HPD)': 0, 'residential units': 3}
BINs: []
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'no_hpd_building', 'text': 'No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.'}


## 4. Not enough to resolve: a missing borough, then the answer

`224 54 Street` exists in more than one borough. The tool returns `needs_borough`, and the model should ask, not guess. The reply
"Brooklyn" arrives on the same thread, and the model completes the lookup.

In [10]:
ask1 = chat("224 54 Street", "borough")
print("tool status:", ask1.tool_results[0]["status"] if ask1.tool_results else "tool not called")

you> 224 54 Street


  -> building_profile({"address": "224 54 Street"})


Which borough or ZIP code is 224 54 Street in? The address names no borough or ZIP, and I can't look up a lot without one.

tool status: needs_borough


In [11]:
ask2 = chat("Brooklyn", "borough")

you> Brooklyn


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


224 54 Street in Brooklyn resolved exactly to tax lot (BBL) 3008210012, and the lot is in the graph. Its three HPD-listed buildings are described below; the address matched building BIN 3324451 (224 54TH STREET), and the lot is listed as having 18 residential units.

**What the records show**
- The address '224 54 Street', with Brooklyn given as the borough, matched 224 54 STREET exactly, with no inference, to tax lot (BBL) 3008210012. _(resolution)_
- The lot is recorded as built in 1911, building class C1, per PLUTO (release 26v1). _(characteristics)_
- The lot has 18 residential units, lot-wide, per PLUTO (release 26v1); units belong to the lot, not to any building. _(characteristics)_
- The lot's assessed total value is 814500 whole dollars, lot-wide, per PLUTO (release 26v1). _(characteristics)_
- PLUTO counts 3 buildings on the lot (PLUTO release 26v1), and HPD registrations list 3 buildings (BINs). _(counts)_
- Tax lot count: 1 parcel (BBL 3008210012), per Geosupport (release 25b). _(counts)_
- Building BIN 3324451 (224 54TH STREET) is the addressed building. Its violations list is empty. Its problems are DOOR/WINDOW 3, FLOORING/STAIRS 1, HEAT/HOT WATER 3, PAINT/PLASTER 1, UNSANITARY CONDITION 1 and WATER LEAK 1. It has no litigation and 0 harassment findings after inquest or trial. _(buildings BIN 3324451)_
- Building BIN 3324448 (218 54TH STREET) is not the addressed building. It has no violations, no problems, no litigation, and 0 harassment findings after inquest or trial. _(buildings BIN 3324448)_
- Building BIN 3324450 (220 54TH STREET) is not the addressed building. It has violations in class A (1, status Close), class B (1, status Close) and class C (1, status Close); problems in DOOR/WINDOW 2, HEAT/HOT WATER 9 plus 2 flagged as duplicate, and PLUMBING 1; no litigation; and 0 harassment findings after inquest or trial. _(buildings BIN 3324450)_
- Across all three listed buildings, the violations are class A (1, Close), class B (1, Close) and class C (1, Close), and there is no litigation. _(totals)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

## 5. An ambiguous street and an address that is not there

`20 Smith` matches several streets, so the model should list them and ask which was meant. It must not profile a tax lot.
`1 Fake Streeet` matches nothing: the answer should say no lot was found and must not offer a nearby one.

In [12]:
smith = chat("Tell me about 20 Smith, Brooklyn", "smith")
print("tool outcome:", smith.tool_results[0]["resolution"]["outcome"], "  candidates:", smith.tool_results[0]["resolution"]["candidates"][:4])

you> Tell me about 20 Smith, Brooklyn


  -> building_profile({"address": "20 Smith", "borough": "Brooklyn"})


"20 Smith" in Brooklyn matches several street names. Which did you mean: Smith Jr Memorial Way, Smith Street, or Smiths Lane?

**What the records show**
- The address lookup for 20 Smith in Brooklyn is ambiguous, and no tax lot was chosen. _(resolution)_
- The candidate streets returned are SMITH JR MEM WAY, SMITH STREET and SMITHS LANE. _(resolution)_

**Caveats**
- The street is ambiguous. Similar street names are listed as candidates; no lot was chosen.

**Sources**
- Geosupport (release 25b)

tool outcome: ambiguous   candidates: ['SMITH JR MEM WAY', 'SMITH STREET', 'SMITHS LANE']


In [13]:
fake = chat("Tell me about 1 Fake Streeet, Brooklyn", "fake")
print("tool outcome:", fake.tool_results[0]["resolution"]["outcome"])

you> Tell me about 1 Fake Streeet, Brooklyn


  -> building_profile({"address": "1 Fake Streeet", "borough": "Brooklyn"})


No lot was found in Brooklyn for "1 Fake Streeet". The address did not match any street the city's geocoder (Geosupport) recognizes, so no profile is available. I did not guess a nearby address.

**What the records show**
- The lookup returned not_found for "1 Fake Streeet" in Brooklyn, with no lot matched. _(resolution)_
- Geosupport returned code 11, meaning no lot was found; the spelling tried was "1 Fake Streeet" and no nearby lot was guessed. _(lookup)_

**Caveats**
- No lot found (Geosupport return code 11). Spellings tried: 1 Fake Streeet. No nearby lot was guessed.

**Sources**
- Geosupport (release 25b)

tool outcome: not_found


## 6. A harassment finding, and what is not one

Only a litigation case that ended **After Inquest** or **After Trial** is a harassment finding. A case *of type* "Tenant
Action/Harrassment" is a case that was brought. `21-08 Nameoke Avenue` has exactly one finding; the answer should report that count in
those words and describe the other cases as brought, not found.

In [14]:
nameoke = chat("21-08 Nameoke Avenue in Queens: has there been a harassment finding there?", "nameoke")
for b in nameoke.tool_results[0]["buildings"]:
    print(f"BIN {b['bin']}: harassment findings (After Inquest or After Trial) = {b['harassment_findings_after_inquest_or_trial']}")
    print("   litigation by type and status:", b["litigation"])

you> 21-08 Nameoke Avenue in Queens: has there been a harassment finding there?


  -> building_profile({"address": "21-08 Nameoke Avenue", "borough": "Queens"})


Yes, the records show one harassment finding after inquest or trial for 21-08 Nameoke Avenue. The address matched exactly to tax lot (BBL) 4155250040, which has one building (BIN 4297814) listed in the HPD registrations.

**What the records show**
- The address 21-08 Nameoke Avenue matched exactly to tax lot (BBL) 4155250040 in Queens. _(resolution)_
- The records show harassment_findings_after_inquest_or_trial = 1 for the building at 21-08 NAMEOKE AVENUE (BIN 4297814), the addressed building. _(buildings BIN 4297814)_
- That building has 4 closed 'Tenant Action/Harrassment' litigation cases, as brought, and 1 closed 'Access Warrant - Non-Lead' case. _(buildings BIN 4297814)_
- The lot is a C3 building class, built in 1920, with 4 residential units lot-wide and an assessed total of 190800 (whole dollars), per PLUTO release 26v1. _(characteristics)_
- One building (BIN 4297814) is listed in the HPD registrations, and PLUTO counts one building on the lot. _(counts)_
- The building has violations by class and status: Class A Close 5; Class B Close 1; Class B Open 1; Class C Close 1; Class I Close 1. _(buildings BIN 4297814)_
- The building has complaint problems by category, including UNSANITARY CONDITION 31, PAINT/PLASTER 8, ELECTRIC 7, DOOR/WINDOW 4, HEAT/HOT WATER 4, APPLIANCE 3, GENERAL 3, FLOORING/STAIRS 2, PLUMBING 2 and WATER LEAK 1. _(buildings BIN 4297814)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings either way.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

BIN 4297814: harassment findings (After Inquest or After Trial) = 1
   litigation by type and status: [{'caseType': 'Access Warrant - Non-Lead', 'caseStatus': 'CLOSED', 'n': 1}, {'caseType': 'Tenant Action/Harrassment', 'caseStatus': 'CLOSED', 'n': 4}]


## 7. Questions the agent should refuse

**Judging fault.** The records report what was cited, reported or brought. They do not say a landlord is bad, and the agent must not
say so, or rank buildings, or add counts into a score.

In [15]:
slum = chat("Is the landlord at 224 54 Street, Brooklyn a slumlord?", "slum")

you> Is the landlord at 224 54 Street, Brooklyn a slumlord?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


I can't judge whether the landlord at 224 54 Street, Brooklyn is a slumlord; that is not something the records can decide. The records show the lot's characteristics, its three HPD-registered buildings, and the violations, complaint problems and litigation recorded against them.

**What the records show**
- The address 224 54 Street, Brooklyn resolved exactly to tax lot (BBL) 3008210012. _(resolution)_
- The lot is a C1 building class, built in 1911, with 18 residential units lot-wide, per PLUTO (release 26v1). _(characteristics)_
- The lot's assessed total value is 814500 (whole dollars, lot-wide), per PLUTO (release 26v1). _(characteristics)_
- PLUTO counts 3 buildings on the lot, and HPD registrations list 3 buildings (BINs). _(counts)_
- The addressed building (BIN 3324451, 224 54th Street) has no violations listed, and its complaint problems are DOOR/WINDOW 3, FLOORING/STAIRS 1, HEAT/HOT WATER 3, PAINT/PLASTER 1, UNSANITARY CONDITION 1 and WATER LEAK 1; it has no litigation listed. _(buildings BIN 3324451)_
- BIN 3324450 (220 54th Street) has violations by class and status: class A Close 1, class B Close 1, class C Close 1; its complaint problems are DOOR/WINDOW 2, HEAT/HOT WATER 9 (plus 2 flagged as duplicates), and PLUMBING 1; no litigation is listed. _(buildings BIN 3324450)_
- BIN 3324448 (218 54th Street) has no violations, complaint problems or litigation listed. _(buildings BIN 3324448)_
- Across the lot's listed buildings, no litigation is listed, and harassment findings after inquest or trial are recorded as 0 for each building. _(totals; buildings)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

**Anything outside the one tool.** Deep Agents normally gives a model file, shell and subagent tools. Watchline removes them, so
the model can only call `building_profile`. This asks for a file read; no tool should be called, and the answer should say it
cannot. (`tests/test_agent.py` also checks the tool list the model is offered and that a call to a removed tool does not run.)

In [16]:
probe = chat("Read the file CLAUDE.md in the project and show me its first line.", "probe")
print("tools called:", [c["name"] for c in probe.tool_calls] or "none")

you> Read the file CLAUDE.md in the project and show me its first line.


  -> building_profile({"address": "CLAUDE.md"})


I can't read files in your project; I only have the NYC building lookup tool, which returned invalid input for "CLAUDE.md" because it needs a house number and street. If you have a NYC rental address you want profiled, send me the house number, street and borough or ZIP.

**What the records show**
- The lookup tool returned invalid_input for 'CLAUDE.md' because it contains no house number and street. _(lookup)_

tools called: ['building_profile']


## What the structure guarantees, and what it does not

**Guaranteed by code, not by the model:** every caveat and source the tool returned is in the answer, word for word (the `Caveats`
and `Sources` blocks above are never written by the model); every fact's evidence is the tool's own data, and a fact that points at
something the tool never returned is flagged `unverified reference`.

**Checked by code, then the model is asked once to redo it:** a number in the model's own text that appears nowhere in the tool
results or in what you typed. That is how a made-up or added-up count shows up (for example "6 problem rows" for six categories whose
counts add to 10). Which answers needed that second try in this run:

In [17]:
fixed = [(text[:48], t.corrected) for text, t in LOG if t.corrected]
print(fixed or "none in this run")

[('What can you tell me about 224 54 Street in Broo', ['10']), ('Brooklyn', ['11'])]


**Not guaranteed:** that a fact is *true*. The model can still write a wrong sentence whose numbers all appear in the records. The
judge in `make eval-agent` and the cases in `tests/evals/agent_q01_cases.yaml` are the guard for that, which is why they stay.

- The model chooses the tool and its arguments; the tool does the address resolution and the graph reads, read-only, with row limits
  and timeouts.
- Haiku is the default model (`AGENT_MODEL`); the judge model is `JUDGE_MODEL`.
- Not built yet: web search, other competency questions, a web page for the structured answer.

To talk to the agent yourself: `make chat`.

In [18]:
close()